# 03 · Load into PostGIS

Same code as `make load`. It creates schema `<slug>` in database `urban_access` and loads every raw
GeoParquet with `ogr2ogr`, reprojected to the analysis CRS. It also loads two prepared tables:
`block_groups` (scored block groups joined with ACS) and `service_points` (destinations from
`classify_places` + GTFS stops). Then it runs `sql/10_prepare.sql` (repair, jsonb, indexes).
Idempotent: every table is dropped and reloaded.

In [ ]:
import os

import urban_access  # noqa: F401  (first: sets PROJ_DATA before geopandas/pyogrio load)
from urban_access.config import load_city

CITY = os.environ.get("CITY", "carson_city")
cfg = load_city(CITY)
print(cfg.name, "| Overture", cfg.overture_release, "| ACS", cfg.acs_year)
print("schema:", cfg.slug, "| EPSG", cfg.analysis_epsg)

In [ ]:
import geopandas as gpd
import pandas as pd
from sqlalchemy import text

from urban_access import db

eng = db.engine()


def q(sql: str, **params) -> pd.DataFrame:
    with eng.connect() as con:
        return pd.read_sql(text(sql.replace("__S__", f'"{cfg.slug}"')), con, params=params)


def qgeo(sql: str) -> gpd.GeoDataFrame:
    with eng.connect() as con:
        return gpd.read_postgis(text(sql.replace("__S__", f'"{cfg.slug}"')), con, geom_col="geom")

In [ ]:
timings = db.load_all(cfg)
pd.Series(timings, name="seconds").round(1)

## Verification: row counts, SRID, geometry validity

In [ ]:
expected = {t: len(gpd.read_parquet(cfg.raw_dir / f)) for t, (f, _) in db.TABLES.items()}
expected["block_groups"] = len(gpd.read_parquet(cfg.interim_dir / "block_groups.parquet"))
expected["service_points"] = len(gpd.read_parquet(cfg.interim_dir / "service_points.parquet"))
rows = []
for t, n in expected.items():
    r = q(f"""SELECT count(*) n, min(ST_SRID(geom)) srid, max(ST_SRID(geom)) srid_max,
                  count(*) FILTER (WHERE NOT ST_IsValid(geom)) invalid,
                  count(*) FILTER (WHERE geom IS NULL) null_geom
           FROM __S__.{t}""").iloc[0]
    rows.append({"table": t, "parquet": n, **r.to_dict()})
check = pd.DataFrame(rows).set_index("table")
assert (check["n"] == check["parquet"]).all(), "row count mismatch"
assert (check["srid"] == cfg.analysis_epsg).all() and (check["srid_max"] == cfg.analysis_epsg).all()
assert (check["invalid"] == 0).all() and (check["null_geom"] == 0).all()
check

In [ ]:
print(q("SELECT service, source, count(*) n FROM __S__.service_points GROUP BY 1, 2 ORDER BY 1"))
bg = qgeo("SELECT geoid, pop_total, geom FROM __S__.block_groups")
print(len(bg), "block groups, population", int(bg["pop_total"].sum()))
assert len(bg) == 46 or cfg.slug != "carson_city"